In [ ]:
import pandas as pd

df_demo = pd.read_csv("/Users/souadmouajel/Desktop/Ironhack/lab-sessions/week-5/day-1/vanguar_ab_test/Data/raw/df_final_demo.txt")
df_demo.info()

In [ ]:
#Cleaning the gender column --------------------------------------gendr-----------------------------------------
#use dictionary for map
map_gender = {
    'm': 'male',
    'male': 'male',
    'f': 'female',
    'female': 'female'
}
#convert to object and nulls are unknow:
df_demo['gendr'] = (
    df_demo['gendr']
        .astype(str)
        .str.strip()
        .str.lower()
        .map(map_gender)
        .fillna('unknown')
)

In [ ]:
df_durations = pd.read_csv("/Users/souadmouajel/Desktop/Ironhack/lab-sessions/week-5/day-1/vanguar_ab_test/Data/clean/durations.csv")
df_durations.info()

In [ ]:
# Test whether the average age of clients engaging with the test group is the same as those engaging with the control group
# Null Hypothesis (H₀):
# There is no difference in the average age between clients in the Test and Control groups.

# 𝜇 Test = 𝜇 Control

 
# Alternative Hypothesis (H₁):
# There is a difference in the average age between clients in the Test and Control groups.

# 𝜇 Test ≠ 𝜇 Control

import pandas as pd
from scipy.stats import ttest_ind

# STEP 1: Inner join on client_id to get age
merged_df = df_durations.merge(df_demo[['client_id', 'clnt_age']], on='client_id', how='inner')

# STEP 2: Remove rows with missing age
merged_df = merged_df.dropna(subset=['clnt_age'])

# STEP 3: Split into test and control groups
test_group = merged_df[merged_df['Variation'] == 'Test']['clnt_age']
control_group = merged_df[merged_df['Variation'] == 'Control']['clnt_age']

# STEP 4: Perform two-sample t-test
t_stat, p_value = ttest_ind(test_group, control_group, equal_var=False)

# STEP 5: Print results
print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.4f}")

# STEP 6: Interpret result at 5% significance
alpha = 0.05
if p_value < alpha:
    print("✅ Reject the null hypothesis: Average age is significantly different between Test and Control groups.")
else:
    print("❌ Fail to reject the null hypothesis: No significant difference in average age between groups.")


In [ ]:
# Test whether the average age of clients in control group is greater than those engaging with test group

# Null hypothesis (H₀):
# 𝜇 Control ≤ 𝜇 Test
# (The Control group’s average age is less than or equal to the Test group’s average age.)

# Alternative hypothesis (H₁):
# 𝜇 Control > 𝜇 Test
# (The Control group’s average age is greater than the Test group’s average age.)
# Assume merged_df as before with 'clnt_age' and 'Variation'

test_group = merged_df[merged_df['Variation'] == 'Test']['clnt_age']
control_group = merged_df[merged_df['Variation'] == 'Control']['clnt_age']

# Perform Welch's t-test (two-tailed)
t_stat, p_value_two_tailed = ttest_ind(control_group, test_group, equal_var=False)

# Convert two-tailed p-value to one-tailed:
# Since we want to test if control mean > test mean,
# check sign of t_stat and adjust p-value accordingly.
if t_stat > 0:
    p_value_one_tailed = p_value_two_tailed / 2
else:
    p_value_one_tailed = 1 - (p_value_two_tailed / 2)

print(f"T-statistic: {t_stat:.3f}")
print(f"One-tailed p-value: {p_value_one_tailed:.4f}")

alpha = 0.05
if p_value_one_tailed < alpha:
    print("✅ Reject H0: Control group's average age is significantly higher than Test group's.")
else:
    print("❌ Fail to reject H0: No significant evidence that Control group is older than Test group.")

In [ ]:
# Test whether the proportion of males and females differs between the Test and Control groups
# Hypothesis: 
# H₀: Gender distribution is independent of the group (Test or Control)
# H₁: Gender distribution depends on the group
from scipy.stats import chi2_contingency

# Merge df_durations and df_demo to get Variation and gender ('gendr')
merged_df = df_durations.merge(df_demo[['client_id', 'gendr']], on='client_id', how='inner')

# Drop rows where gender is missing or 'unknown' (case-insensitive)
merged_df = merged_df.dropna(subset=['gendr'])
merged_df = merged_df[~merged_df['gendr'].str.lower().isin(['unknown'])]

# Create contingency table
contingency_table = pd.crosstab(merged_df['Variation'], merged_df['gendr'])

print("Contingency Table:")
print(contingency_table)

# Perform Chi-square test
chi2, p, dof, expected = chi2_contingency(contingency_table)

print(f"\nChi-square statistic: {chi2:.3f}")
print(f"P-value: {p:.4f}")

alpha = 0.05
if p < alpha:
    print("✅ Reject H0: Gender distribution differs significantly between Test and Control groups.")
else:
    print("❌ Fail to reject H0: No significant difference in gender distribution between groups.")